# ThetaSwap: Fee Concentration Hedge Demo

**What if you could hedge against JIT competition?**

This notebook demonstrates how ThetaSwap's fee concentration derivative protects passive LPs against adverse competition from just-in-time (JIT) liquidity providers.

We simulate three market regimes:
1. **Calm** — Low concentration, competitive equilibrium
2. **Gradual JIT Growth** — A JIT bot ramps up over 1000 blocks
3. **Shock** — Sudden MEV event spikes concentration

For each, we compare a hedged PLP (using ThetaSwap) vs an unhedged PLP.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from simulation.config import SimConfig
from simulation.scenarios import run_simulation
from simulation.metrics import hedge_effectiveness, ecosystem_welfare
from simulation.plotting import (
    plot_pnl_comparison,
    plot_index_and_price,
    plot_reserves,
    plot_funding_and_premium,
    plot_welfare_summary,
)

%matplotlib inline
plt.rcParams["figure.dpi"] = 120

## Setup

We use the default simulation parameters: 3000 blocks across three phases, 30 bps base fee, 10% premium fraction. The fee concentration index $A_T$ evolves synthetically through calm → gradual → shock.

In [ ]:
cfg = SimConfig()
result = run_simulation(cfg, seed=42, scenario="narrative")

print(f"Simulation: {cfg.T} blocks, L_0={cfg.L_0}, alpha={cfg.alpha}")
print(f"Fee: {cfg.fee_base*10000:.0f} bps base, {cfg.fee_max*10000:.0f} bps max")
print(f"Premium fraction: {cfg.premium_fraction*100:.0f}%")
print(f"A_T range: [{result.A_T.min():.3f}, {result.A_T.max():.3f}]")

## Plot 1: The Money Plot — Hedged vs Unhedged P&L

The green line shows the hedged PLP's cumulative P&L. The red dashed line shows the unhedged PLP. Background shading indicates the three market regimes.

Notice how the lines track closely during the calm phase (hedge costs little), diverge during gradual JIT growth, and dramatically separate during the shock.

In [ ]:
fig = plot_pnl_comparison(result)
plt.show()

## Plot 2: Fee Concentration Index & Price Response

The top panel shows $A_T$ — the fee concentration index — evolving through the three phases. The bottom panel shows how the CFMM's index price $p_{index} = A_T/(1-A_T)$ and mark price $p_{mark}$ track each other.

The shaded area between them is the *basis* — the signal that drives funding rate payments.

In [ ]:
fig = plot_index_and_price(result)
plt.show()

## Plot 3: Reserve Composition

The CFMM starts fully in the risky asset (at $A_0 = 0$). As fee concentration increases, reserves shift from risky to numeraire — this is the protection payoff mechanism in action.

The dotted line shows total LP value $px + y$ tracking $V_{LP}(p) = \ln(1+p)$.

In [ ]:
fig = plot_reserves(result)
plt.show()

## Plot 4: Funding Rate & Premium Cost

The top panel shows the funding rate and dynamic fee. The bottom panel shows cumulative premium flows: what the PLP pays vs what the underwriter earns.

Key insight: the cost of the hedge is small during calm markets and only scales when protection is actually needed.

In [ ]:
fig = plot_funding_and_premium(result)
plt.show()

## Plot 5: Ecosystem Welfare

This bar chart shows final P&L for each participant. The hedge is **not zero-sum**: total welfare (PLP hedged + Underwriter) is positive because the hedge enables PLPs to remain in the pool and earn more total fees.

In [ ]:
fig = plot_welfare_summary(result)
plt.show()

## Summary Statistics

In [ ]:
eff = hedge_effectiveness(result.hedged_pnl[-1], result.unhedged_pnl[-1])
welf = ecosystem_welfare(result.hedged_pnl[-1], result.underwriter_pnl[-1])

print("=" * 50)
print("FINAL RESULTS")
print("=" * 50)
print(f"Hedged PLP P&L:    {result.hedged_pnl[-1]:>10.2f}")
print(f"Unhedged PLP P&L:  {result.unhedged_pnl[-1]:>10.2f}")
print(f"Underwriter P&L:   {result.underwriter_pnl[-1]:>10.2f}")
print(f"Hedge Effectiveness: {eff:>8.1%}")
print(f"Ecosystem Welfare:   {welf:>10.2f}")
print(f"Premium Paid:        {result.cum_premium_paid[-1]:>10.2f}")
print(f"Max A_T:             {result.A_T.max():>10.3f}")